# Reframe frequency as classification — reproduce §14.14 (issue #67)

**What this reproduces:** the current branch's experiment — Spanish motor claim counts
(`N_claims_year`) re-expressed as **binary** (claim vs no-claim, 11.1% positive) and **ordinal**
(0/1/2+), scored on the multiclass-native metric set. Protocol: 5-fold, seeds 42 + 7 (binary),
seed 42 (ordinal), 9 methods (binary) / 6 methods (ordinal), leak-fixed load, TabPFN
`model_path="v3_default"` with the canonical retry loop.

**How it works:** calls the exact script that produced the committed results —
`scripts/eval/insurance_benchmark_v1/run_reframe_frequency.py` (no CLI args by design).

**Requirements:** benchmark-venv kernel + one-click **browser login** in the preflight cell (token saved to the gitignored repo-root `.env`), or `TABPFN_API_KEY` set manually — see `notebooks/reproducibility/README.md`.

**Cost:** ~120 fold-rows total; the hosted-API TabPFN arms dominate. Budget 20–60 minutes.

**⚠ Warning:** re-running **overwrites the committed** `reframe_frequency_results.csv`
(and the summary CSV if the analysis cell is run). For exploration, copy the CSVs aside first.

**Expected verdict (§14.14):** TabPFN ranks #1 on every metric, but several margins are within
noise (binary PR-AUC p=0.12, ordinal Brier p=0.88 — the report's own §14.14.6 honest weak spot);
significant margins are seed-stable (binary AUC 0.7170, +0.0080 vs LightGBM, p=0.0010) — while
the same rows scored as a count model had TabPFN *behind* LGBM on Poisson deviance (§14.9).
The count axis was the loss; classification is the win.

## Shipped with executed evidence

This notebook ships with its evidence cells already executed — the fold-table and the binary/ordinal summaries below are the committed canonical results. Read the outputs to follow along, or re-run any cell: everything except the two subprocess cells works instantly from the committed `reframe_frequency_results.csv`.

**The optional cells (tagged `optional`):** the 'run the experiment' cell re-runs the hosted API (20–60 min, costs credits, overwrites the committed CSV) and the paired-stats cell re-runs the analysis layer (overwrites `reframe_frequency_summary.csv`). Both were skipped here — every other cell is safe and cheap to re-run.

In [1]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *[Path.cwd().parents[i] for i in range(1, 4)]]
            if (p / "scripts/eval/insurance_benchmark_v1/run_reframe_frequency.py").exists())
print("repo root:", ROOT)

repo root: /Users/Scott/Documents/Data Science/ADSWP/TabPFN-work-scott-worktrees/cumans


In [2]:
# Preflight: versions + auth. No TABPFN_API_KEY configured? One-click browser login:
# opens the Prior Labs login page, then saves the token to the repo-root .env
# (gitignored) and exports it for this session's subprocesses.
import importlib, os
for m in ("tabpfn_client", "pandas", "sklearn"):
    mod = importlib.import_module(m)
    print(f"{m:14s} {getattr(mod, '__version__', '?')}")

def _have_key() -> bool:
    if os.environ.get("TABPFN_API_KEY"):
        return True
    env = ROOT / ".env"
    return env.exists() and any(l.startswith("TABPFN_API_KEY=") for l in env.read_text().splitlines())

if not _have_key():
    from tabpfn_client.browser_auth import BrowserAuthHandler
    ok, token = BrowserAuthHandler().try_browser_login()
    assert ok and token, "Browser login failed — see notebooks/reproducibility/README.md"
    env = ROOT / ".env"
    lines = [l for l in env.read_text().splitlines() if not l.startswith("TABPFN_API_KEY=")] if env.exists() else []
    lines.append(f"TABPFN_API_KEY={token}")
    env.write_text("\n".join(lines) + "\n")
    os.environ["TABPFN_API_KEY"] = token
    print("Browser login OK — token saved to repo-root .env (gitignored).")
print("API key present:", _have_key())
assert _have_key(), "no API key — rerun this cell to trigger browser login, or set TABPFN_API_KEY manually"

tabpfn_client  ?
pandas         2.3.3
sklearn        1.6.1
API key present: True


In [ ]:
# OPTIONAL — re-run the experiment (hosted API: 20–60 min, costs credits, overwrites the committed results).
# The exact command behind §14.14 (no args — the protocol is hardcoded in the script).
import subprocess, time

cmd = [sys.executable, "scripts/eval/insurance_benchmark_v1/run_reframe_frequency.py"]
print("running:", " ".join(cmd))
t0 = time.time()
r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-5000:])
print(f"exit: {r.returncode}  ({time.time() - t0:.0f}s)")
if r.returncode:
    print(r.stderr[-2000:])

In [3]:
# The evidence: 120 fold-rows (5 folds × (9 methods × 2 binary seeds + 6 methods × 1 ordinal seed)).
import pandas as pd
res = pd.read_csv(ROOT / "scripts/eval/insurance_benchmark_v1/reframe_frequency_results.csv")
print("rows:", len(res), "| tasks:", res.task.unique(), "| seeds:", sorted(res.seed.unique()))
res.head(3).T

rows: 120 | tasks: ['binary' 'ordinal'] | seeds: [7, 42]


,0,1,2
dataset,spanish_motor_freq,spanish_motor_freq,spanish_motor_freq
task,binary,binary,binary
method,cat,cat,cat
fold,0,1,2
seed,42,42,42
log_loss,0.324211,0.320857,0.319891
auc,0.697599,0.711989,0.712839
brier,0.09345,0.092852,0.092408
pr_auc,0.236663,0.237818,0.247402
lift10,2.462415,2.454011,2.680672


In [4]:
# Binary task — mean over folds, per method and seed. TabPFN should top AUC (and PR-AUC) both seeds.
bin_cols = ["log_loss", "auc", "pr_auc", "brier", "lift10"]
res[res.task == "binary"].groupby(["method", "seed"])[bin_cols].mean().round(4).sort_values("auc", ascending=False)

log_loss     auc  pr_auc   brier  lift10
method      seed                                          
tabpfn      42      0.3206  0.7170  0.2428  0.0927  2.6018
            7       0.3209  0.7165  0.2411  0.0928  2.5312
lgbm        42      0.3209  0.7090  0.2365  0.0929  2.4976
cat         7       0.3219  0.7078  0.2366  0.0931  2.4892
            42      0.3220  0.7070  0.2374  0.0931  2.5127
lgbm        7       0.3217  0.7067  0.2335  0.0931  2.4505
rf          7       0.3475  0.6898  0.2227  0.0944  2.3682
            42      0.3472  0.6892  0.2211  0.0945  2.3816
xgb         7       0.3325  0.6883  0.2193  0.0955  2.3715
            42      0.3337  0.6862  0.2164  0.0958  2.3060
lr          7       0.3476  0.5817  0.1408  0.0987  1.4219
            42      0.3512  0.5663  0.1365  0.0994  1.3799
logisticglm 42      0.3510  0.5641  0.1329  0.0995  1.3026
            7       0.3532  0.5547  0.1308  0.0999  1.2874
poissonglm  7       0.3490  0.5000  0.1112  0.0988  0.9412
tweedieglm  7       0.3490  0.5000  0.1112  0.0988  0.9412
            42      0.3490  0.5000  0.1112  0.0988  0.9395
poissonglm  42      0.3490  0.5000  0.1112  0.0988  0.9395

In [5]:
# Ordinal task — one-vs-rest macro AUC, multiclass log loss / Brier, lift10 on P(>=1).
# PR-AUC is NaN by design (no standard multiclass PR-AUC) — lift10 is the substitute.
ord_cols = ["auc", "log_loss", "brier", "lift10"]
res[res.task == "ordinal"].groupby("method")[ord_cols].mean().round(4).sort_values("auc", ascending=False)

,auc,log_loss,brier,lift10
method,,,,
tabpfn,0.7167,0.3935,0.0647,2.7370
lgbm,0.7056,0.3944,0.0647,2.6491
cat,0.7035,0.3967,0.0649,2.6444
xgb,0.6889,0.4061,0.0657,2.5554
rf,0.6838,0.4966,0.0659,2.5449
lr,0.5507,0.4335,0.0690,1.2215


In [ ]:
import subprocess

# OPTIONAL — re-run the analysis layer (writes reframe_frequency_summary.csv).
# Optional: the paired-stats layer (deltas + p-values vs best GLM) that the report quotes.
# Re-running this also overwrites reframe_frequency_summary.csv.
r2 = subprocess.run([sys.executable, "scripts/eval/insurance_benchmark_v1/analyze_reframe_frequency.py"],
                    cwd=ROOT, capture_output=True, text=True)
print(r2.stdout[-4000:])
if r2.returncode:
    print(r2.stderr[-2000:])

## Reading the verdict

- **Binary (seed 42):** TabPFN AUC 0.7170 (+0.0080 vs LightGBM, p=0.0010); rank #1 on all five
  metrics. Seed 7 confirms (0.7165, +0.0098, p=0.0020).
- **Ordinal:** one-vs-rest AUC 0.7167 (+0.0111 vs LightGBM, p=0.0085) — the largest edge of the two reframes.
- **GLM collapse:** poissonglm/tweedieglm predict a constant on claim/no-claim (AUC exactly 0.5000,
  log loss at base rate 0.3490) — their count-domain edge does not transfer.
- **Scoping:** single dataset (spanish_motor_freq); freMTPL2freq (678K rows) was NOT reframed;
  the count model itself (Poisson deviance) and the pricing story are unchanged.

Master report §14.14; learning path S5 / Stage 4.5; digest `docs/MASTER-REPORT-DIGEST.md`.